### MODELAGEM DA INFERENCIA DE CAUSALIDADE DE ACIDENTES DE TRANSITO

 - Usando ordered logit para modelar primeiramente (sem a causa pois ela possui endogeinedade) o que influencia probabilisticamente em um acidente de transito.
 - Para o segundo modelo e feito com a endogeinedade para saber o impacto que as variaveis causam no acidente de transito sem se preocupar com o efeito probabilistico de o quanto digamos 1 km a mais influencia da probabilidade de ser um acidente de alta letalidade. Somente para direcionar politica publica (o onde agir)

In [1]:
import polars as pl
import pandas as pd
import statsmodels.api as sm
from patsy import dmatrix
import warnings

from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.miscmodels.ordinal_model import OrderedModel
warnings.simplefilter('ignore', ConvergenceWarning)

In [2]:
df = pl.read_parquet("./data/anuario_prf.parquet")
df = df.to_pandas()

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 258737 entries, 0 to 258736
Data columns (total 19 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   uf                      258737 non-null  category      
 1   br                      258737 non-null  category      
 2   km                      258737 non-null  int64         
 3   causa_acidente          258737 non-null  category      
 4   tipo_acidente           258737 non-null  category      
 5   fase_dia                258737 non-null  category      
 6   sentido_via             258737 non-null  category      
 7   condicao_metereologica  258737 non-null  category      
 8   tipo_pista              258737 non-null  category      
 9   tracado_via             258737 non-null  category      
 10  uso_solo                258737 non-null  category      
 11  pessoas                 258737 non-null  int64         
 12  veiculos                258737

In [4]:
crash_counts = df['br'].value_counts()

state_counts = df.groupby('br')['uf'].nunique()

/tmp/ipykernel_5033/2834029756.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  state_counts = df.groupby('br')['uf'].nunique()


In [5]:
br_stats = pd.DataFrame({
    'total_acidentes': crash_counts,
    'num_estados': state_counts
})

In [6]:
min_estados = 5
top_brs_stats = br_stats[
    br_stats['num_estados'] >= min_estados
].sort_values(by='total_acidentes', ascending=False)

In [7]:
TOP_BRS_LIST = top_brs_stats.head(5).index.tolist()

In [8]:
print(f"As Top 5 BRs (>= {min_estados} estados e mais acidentes) são:")
print(TOP_BRS_LIST)
print("\nEstatísticas completas das BRs filtradas:")
print(top_brs_stats)

As Top 5 BRs (>= 5 estados e mais acidentes) são:
['101', '116', '153', '163', '364']

Estatísticas completas das BRs filtradas:
     total_acidentes  num_estados
br                               
101            44646           11
116            40240           10
153             9856            8
163             8910            5
364             7961            5
230             5794            7
316             4450            5
20              2741            5
158             2634            8
110             1197            5
235             1144            5
226              996            5
0                669           27


In [9]:
for br in TOP_BRS_LIST:
    df[f'br_{br}'] = (df['br'] == br).astype(int)
    
    df[f'km_br_{br}'] = df['km'] * df[f'br_{br}']

In [10]:
risco_order = pd.CategoricalDtype(
    categories=['baixo', 'medio', 'alto'],
    ordered=True
)
df['risco_ordenado'] = df['risco'].astype(risco_order)

In [11]:
y = df['risco_ordenado'].cat.codes

In [12]:
main_effects = [f"br_{br}" for br in TOP_BRS_LIST]
interaction_effects = [f"km_br_{br}" for br in TOP_BRS_LIST]
all_new_terms = main_effects + interaction_effects
new = " + ".join(all_new_terms)

In [ ]:
def agrupar 

In [13]:
formula_modelo_1 = f"""
    km + {new} + pessoas + veiculos + C(uf) +
    C(fase_dia) + C(condicao_metereologica) +
    C(tipo_pista) + C(tracado_via) + C(uso_solo) +
    C(tipo_acidente) + mes + dia_semana_num
"""

In [14]:
X1 = dmatrix(formula_modelo_1, df, return_type='dataframe')
X1.drop(columns=['Intercept'], inplace=True)

In [15]:
X1

,C(uf)[T.MS],C(uf)[T.RJ],C(uf)[T.MG],C(uf)[T.PB],C(uf)[T.SC],C(uf)[T.CE],C(uf)[T.MT],C(uf)[T.PE],C(uf)[T.BA],C(uf)[T.GO],...,br_364,km_br_101,km_br_116,km_br_153,km_br_163,km_br_364,pessoas,veiculos,mes,dia_semana_num
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,33.0,0.0,0.0,0.0,3.0,2.0,1.0,6.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,393.0,0.0,3.0,3.0,1.0,6.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,457.0,0.0,0.0,0.0,0.0,2.0,2.0,1.0,6.0
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,1.0,6.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,8.0,0.0,0.0,0.0,3.0,2.0,1.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258732,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,9.0,6.0
258733,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,8.0,6.0
258734,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,8.0,1.0
258735,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,321.0,0.0,0.0,0.0,0.0,6.0,2.0,9.0,7.0


In [16]:
model_1 = OrderedModel(y, X1, distr='logit')

In [17]:
fit_1 = model_1.fit(method='bfgs', disp=False)

KeyboardInterrupt: 

In [ ]:
print(fit_1.summary())

In [ ]:
formula_modelo_2 = """
    km + {new} + pessoas + veiculos + C(uf) +
    C(fase_dia) + C(condicao_metereologica) +
    C(tipo_pista) + C(tracado_via) + C(uso_solo) +
    C(tipo_acidente) + mes + dia_semana_num +
    C(causa_acidente)
"""

In [ ]:
X2 = dmatrix(formula_modelo_2, df, return_type='dataframe')
X2.drop(columns=['Intercept'], inplace=True)

In [ ]:
model_2 = OrderedModel(y, X2, distr='logit')

In [ ]:
fit_2 = model_2.fit(method='bfgs', disp=False)

In [ ]:
print(fit_2.summary())